In [ ]:
import os
import copy as cp
import json
from ase.io import read
#import calorine
from calorine.nep import setup_training, get_descriptors
import phonopy
from hiphive.structure_generation import generate_phonon_rattled_structures

## Data pre-processing

In [ ]:
# Define the directory in which to create the input files for NEP training
run_dir = 'results/bulk/BaTiO3/ML'

# Read in the structures and energies
structures = read(os.path.join(run_dir, 'structures.xyz@0:'))

In [ ]:
from src.phononcalc import phonon_to_atoms

In [ ]:
# Load the phonon calculation results from the YAML file
phonon = phonopy.load("results/bulk/BaTiO3/0082/phonons/BaTiO3.yaml")
# Convert the phonon object to an ASE Atoms object
atoms = phonon_to_atoms(phonon, cell='super')
# Produce the force constants
phonon.produce_force_constants()
# Get the force constants matrix
fc = phonon.force_constants
# Generate new structures using the force constants and the original structure
structures_gen = generate_phonon_rattled_structures(atoms, fc2=fc, n_structures=100, temperature=300)

In [ ]:
fc_new = fc.transpose(0, 2, 1, 3)
fc_new = fc_new.reshape(3 * 40, 3 * 40)

In [ ]:
structures_gen = generate_phonon_rattled_structures(atoms, fc2=fc, n_structures=100, temperature=300)

In [ ]:
from ase.visualize import view
view(structures_gen)

In [ ]:
# Load numpy file
import numpy as np
np.load('results/bulk/SrTiO3/0001/frozen/R/mode_1/Q_1/time.npy')


In [ ]:
tests = read('results/bulk/SrTiO3/0001/frozen/R/mode_1/Q_1/structures.xyz@0:')

In [ ]:
len(tests)

In [ ]:
energies = {'Ba': -761.227747,
            'Ti': -1604.974503,
            'O': -440.177463}

with open(os.path.join(run_dir, 'energies.json'), 'w') as f:
    json.dump(energies, f)

In [ ]:
# Load the energies from the JSON file
with open(os.path.join(run_dir, 'energies.json'), 'r') as f:
    energies1 = json.load(f)

energies1

In [ ]:
unique_elements = set()
atom_energies = []
for atoms in structures:
    N_atoms = len(atoms)
    elements = atoms.get_chemical_symbols()
    unique_elements.update(elements)
    atoms.calc.results['energy'] -= sum(energies[element] for element in elements)
    atoms.calc.results['energy'] /= N_atoms
    atom_energies.append(atoms.calc.results['energy'])

N_elements = len(unique_elements)

In [ ]:
structures[0].get_potential_energy()

In [ ]:
' '.join(unique_elements)

In [ ]:
len(structures)

In [ ]:
print(len(structures))
print(type(structures))

In [ ]:
# Shuffle order of structures
import random
random.shuffle(structures)

In [ ]:

# prepare input for NEP construction
parameters = dict(version=4,
                  type=[N_elements, ' '.join(unique_elements)],
                  cutoff=[8, 4],
                  neuron=30,
                  generation=100000,
                  batch=1000000)

setup_training(parameters, structures,
               rootdir='MLtest', overwrite=True,
               mode='kfold', n_splits=10)

In [ ]:
len(read('MLtest/nepmodel_split1/test.xyz@0:'))

## Model analysis

In [ ]:
import numpy as np
import pandas as pd
from ase.units import GPa
from calorine.nep import get_parity_data, read_loss, read_structures
from matplotlib import pyplot as plt
from pandas import DataFrame, concat as pd_concat
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
loss = read_loss('results/bulk/BaTiO3/ML/nepmodel_full/loss.out')

fig, axes = plt.subplots(figsize=(6.0, 4.0), nrows=2,sharex=True, dpi=120)

ax = axes[0]
ax.set_ylabel('Loss')
ax.plot(loss.total_loss, label='total')
ax.plot(loss.L2, label='$L_2$')
ax.plot(loss.L1, label='$L_1$')
ax.set_yscale('log')
ax.legend()

ax = axes[1]
ax.plot(loss.RMSE_F_train, label='forces (eV/Å)')
ax.plot(loss.RMSE_V_train, label='virial (eV/atom)')
ax.plot(loss.RMSE_E_train, label='energy (eV/atom)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Generation')
ax.set_ylabel('RMSE')
ax.legend()

plt.tight_layout()
fig.subplots_adjust(hspace=0, wspace=0)
fig.align_ylabels(axes)

In [ ]:
training_structures, _ = read_structures('results/bulk/BaTiO3/ML/nepmodel_full')

units = dict(
    energy='eV/atom',
    force='eV/Å',
    virial='eV/atom',
    stress='GPa',
)
# Make a 2x2 grid of parity plots for energy, force, virial, and stress
fig, axes = plt.subplots(2, 2, figsize=(6, 5))
kwargs = dict(alpha=0.2, s=0.5)
axes = axes.flatten()

# Loop over the properties and units, get the parity data, calculate R2 and RMSE, and plot the parity plots
for icol, (prop, unit) in enumerate(units.items()):
    df = get_parity_data(training_structures, prop, flatten=True)
    R2 = r2_score(df.target, df.predicted)
    rmse = np.sqrt(mean_squared_error(df.target, df.predicted))

    ax = axes[icol]
    ax.set_xlabel(f'Target {prop} ({unit})')
    ax.set_ylabel(f'Predicted ({unit})')
    ax.scatter(df.target, df.predicted, **kwargs)
    ax.set_aspect('equal')
    mod_unit = unit.replace('eV', 'meV').replace('GPa', 'MPa')
    ax.text(0.1, 0.75, f'{1e3*rmse:.1f} {mod_unit}\n' + '$R^2= $' + f' {R2:.5f}', transform=ax.transAxes)

fig.tight_layout()
plt.show()

## Phonon dispersion

In [ ]:
from ase.io import read
from calorine.calculators import CPUNEP
from calorine.tools import relax_structure, get_force_constants

In [ ]:
structure = read('results/bulk/BaTiO3/0082/frozen/G/mode_1/Q_1/structures.xyz@0:')[0]
calculator = CPUNEP('results/bulk/BaTiO3/ML/nepmodel_full/nep.txt')
structure.calc = calculator
relax_structure(structure, fmax=0.0001)

In [ ]:
phonon = get_force_constants(structure, calculator, [2, 2, 2])

In [ ]:
from src.phononcalc import get_phonon_dispersion, get_phonon_dos, get_phonon_pdos, order_labels
from src.plotsettings import PlotSettings
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

In [ ]:
def plot_dispersion(phonon, bulk=True, pDOS=False):
    """Function to plot the phonon dispersion and DOS together.
    Parameters:
    - phonon: Phonopy object containing phonon data.
    - pDOS: Whether to plot the projected density of states (PDOS) (default is True).
    - bulk: Boolean indicating if the system is bulk (True) or slab (False).
    Returns:
    - None. The function creates a plot of the phonon dispersion and DOS.
    """

    # Define tickmarks for the x- and y-axis
    E_tickmarks = np.arange(-10, 26, 5)
    # Convert tickmarks to strings with i for negative numbers
    E_ticklabels = []
    for tick in E_tickmarks:
        if tick < 0:
            E_ticklabels.append(f'{abs(tick)}i')
        else:
            E_ticklabels.append(f'{tick}')
    
    # Define tickmarks for the x- and y-axis
    dos_tickmarks = np.arange(0, 7, 1)

    # Make a simple figure where graphs are plotted
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 5),
                             sharey='col', gridspec_kw={'width_ratios': [1, 0.4]})
    
    # Define two axes, one for the band structure and one for the DOS
    ax1, ax2 = axes

    plt.subplots_adjust(wspace=0.08)
    PlotSettings().set_size(fig)
    
    def _plot_disp(ax, phonon, val, col='k', vlines=True):
        # Extract phonon dispersion data
        (dist, X, freq, labels) = get_phonon_dispersion(phonon, bulk)
        dist = np.array(dist)
        dist /= dist[-1][-1]  # Normalize distances to the total length of the path
        X /= X[-1]  # Normalize high symmetry point locations to the total length of the path
        if vlines:
            # Plot vertical lines at symmetry points
            ax.vlines(X, E_tickmarks[0], E_tickmarks[-1], color='0.5', lw=0.8)
        # Plot dashed line at 0
        #ax.axhline(y=0, color='k', linestyle=':')
        # Determine the number of segments between symmetry points and the number of modes
        n_segments = len(freq)
        n_modes = freq[0].shape[1]
        # Loop over all segments and modes and plot everything
        for i in range(n_segments):
            for j in range(n_modes):
                if i == 0 and j == 0:
                    ax.plot(dist[i], freq[i][:, j], color=col, lw=1, label=f'{val}')
                else:
                    ax.plot(dist[i], freq[i][:, j], color=col, lw=1)
        # Set x- and y-ticks
        ax.set_xticks(X, labels)
        ax.set_yticks(E_tickmarks, E_ticklabels)
        # Set x- and y-limits
        ax.set_xlim(X[0], X[-1])
        ax.set_ylim(E_tickmarks[0], E_tickmarks[-1])

    def _plot_dos(ax, phonon, val='DOS', col='k'):
        # Extract total DOS data
        (dosx, dosy) = get_phonon_dos(phonon, bulk)
        # Plot total DOS
        ax.plot(dosx, dosy, lw=1, color=col, label=f'{val}')
        if pDOS:
            ax.fill_between(dosx, dosy, color='lightgray', alpha=0.5)

    def _plot_pdos(ax, phonon):
        atom_colors = {'Ba': 'tab:blue', 'Sr': 'tab:purple',
                       'Ti': 'tab:orange', 'O': 'tab:red'}
        # Extract PDOS data
        (pdosx, pdosy, symbols) = get_phonon_pdos(phonon, bulk)
        # Plot PDOS
        for i in range(pdosx.shape[0]):
            ax.plot(pdosx[i], pdosy, lw=1, color=atom_colors[symbols[i]], label=f'{symbols[i]}')
        # Get all handles and labels
        handles, labels = ax.get_legend_handles_labels()
        # Remove duplicates and sort for the legend
        sorted_handles, sorted_labels = order_labels(symbols, handles, labels)
        # Add legend with duplicates removed and sorted labels
        ax.legend(sorted_handles, sorted_labels, loc='best', fontsize=14)

    # Plot dashed line at Fermi level for both subplots
    ax1.axhline(y=0, color='k', linestyle=':', lw=0.8)
    ax2.axhline(y=0, color='k', linestyle=':', lw=0.8)

    # Plot phonon dispersion and total DOS for PW
    _plot_disp(ax1, phonon, 'ML', col='k')
    _plot_dos(ax2, phonon, col='k')
    if pDOS:
        # Plot PDOS for PW
        _plot_pdos(ax2, phonon)

    # Set x- and y-label
    ax1.set_xlabel('k-points')
    ax1.set_ylabel('Frequency (THz)')
    # Add minor tickmarks to the y-axis
    ax1.yaxis.set_minor_locator(AutoMinorLocator())
    
    ax2.set_xlabel('DOS (1/THz)')
    ax2.legend(loc='upper right')
    # Force x- and y-ticks
    ax2.set_xticks(dos_tickmarks, dos_tickmarks)
    ax2.set_yticks(E_tickmarks, E_ticklabels)
    # Set limits to match
    ax2.set_xlim(dos_tickmarks[0], dos_tickmarks[-1])
    ax2.set_ylim(E_tickmarks[0], E_tickmarks[-1])
    # Hide y-tick labels
    ax2.set_yticklabels([])
    
    # Show figure
    plt.show()

In [ ]:
plot_dispersion(phonon, bulk=True, pDOS=False)

## Testing

In [ ]:
import os
import random
import subprocess
import numpy as np

import phonopy
from hiphive.structure_generation import generate_phonon_rattled_structures

from ase.io import write
from ase.neighborlist import neighbor_list
from ase.data import covalent_radii

from calorine.nep import setup_training


class ActiveLearningNEP:

    def __init__(self,
                 yaml_file,
                 calculator_function,
                 parameters,
                 elements,
                 run_dir="runs",
                 temperatures=[100,300,600],
                 n_generate=50,
                 descriptor_threshold=0.5):

        self.yaml_file = yaml_file
        self.calc_fn = calculator_function
        self.parameters = parameters
        self.elements = elements
        self.run_dir = run_dir

        self.temperatures = temperatures
        self.n_generate = n_generate
        self.descriptor_threshold = descriptor_threshold

        self.dataset = []
        self.training_descriptors = []

        os.makedirs(run_dir, exist_ok=True)

        # load phonons
        self.phonon = phonopy.load(yaml_file)
        self.phonon.produce_force_constants()

        self.fc = self.phonon.force_constants
        self.atoms = phonon_to_atoms(self.phonon, cell="super")

    # ------------------------------------------------
    # STRUCTURE GENERATION
    # ------------------------------------------------

    def generate_structures(self):

        seed_structure = random.choice(self.dataset) if self.dataset else self.atoms

        structures = []

        for T in self.temperatures:

            new_structs = generate_phonon_rattled_structures(
                seed_structure,
                fc2=self.fc,
                n_structures=self.n_generate,
                temperature=T
            )

            structures.extend(new_structs)

        return structures

    # ------------------------------------------------
    # SANITY FILTER
    # ------------------------------------------------

    def structure_filter(self, atoms_list, factor=0.7):

        filtered = []

        for atoms in atoms_list:

            numbers = atoms.get_atomic_numbers()

            i, j, d = neighbor_list("ijd", atoms, cutoff=3.0)

            valid = True

            for ii, jj, dist in zip(i, j, d):

                rmin = factor * (
                    covalent_radii[numbers[ii]] +
                    covalent_radii[numbers[jj]]
                )

                if dist < rmin:
                    valid = False
                    break

            if valid:
                filtered.append(atoms)

        return filtered

    # ------------------------------------------------
    # DFT CALCULATIONS
    # ------------------------------------------------

    def run_dft(self, atoms_list):

        results = []

        for atoms in atoms_list:

            atoms = self.calc_fn(atoms)
            results.append(atoms)

        return results

    # ------------------------------------------------
    # NEP TRAINING
    # ------------------------------------------------

    def train_nep(self, iteration):

        run_path = os.path.join(self.run_dir, f"iter_{iteration}")

        setup_training(self.parameters,
                       self.dataset,
                       rootdir=run_path,
                       overwrite=True)

        subprocess.run(
            ["nep"],
            cwd=os.path.join(run_path, "nepmodel_full"),
            check=True
        )

        return run_path

    # ------------------------------------------------
    # DESCRIPTOR EXTRACTION
    # ------------------------------------------------

    def compute_descriptor(self, atoms):

        # placeholder: replace with NEP descriptor if available
        pos = atoms.get_positions()

        return pos.flatten()

    # ------------------------------------------------
    # DESCRIPTOR DISTANCE
    # ------------------------------------------------

    def descriptor_distance(self, descriptor):

        if not self.training_descriptors:
            return 0

        dists = []

        for d in self.training_descriptors:

            dists.append(np.linalg.norm(descriptor - d))

        return min(dists)

    # ------------------------------------------------
    # STRUCTURE SELECTION
    # ------------------------------------------------

    def select_structures(self, structures):

        selected = []

        for atoms in structures:

            desc = self.compute_descriptor(atoms)

            dist = self.descriptor_distance(desc)

            if dist > self.descriptor_threshold:
                selected.append(atoms)

        return selected

    # ------------------------------------------------
    # UPDATE DESCRIPTOR DATABASE
    # ------------------------------------------------

    def update_descriptors(self):

        self.training_descriptors = [
            self.compute_descriptor(a) for a in self.dataset
        ]

    # ------------------------------------------------
    # MAIN LOOP
    # ------------------------------------------------

    def run(self, n_iterations=10):

        print("Generating initial dataset")

        structures = self.generate_structures()

        structures = self.structure_filter(structures)

        new_data = self.run_dft(structures)

        self.dataset += new_data

        self.update_descriptors()

        write("dataset.xyz", self.dataset)

        for iteration in range(n_iterations):

            print(f"\n--- Iteration {iteration} ---")

            self.train_nep(iteration)

            candidates = self.generate_structures()

            candidates = self.structure_filter(candidates)

            selected = self.select_structures(candidates)

            print("Selected:", len(selected))

            if not selected:
                print("No new structures selected")
                break

            new_data = self.run_dft(selected)

            self.dataset += new_data

            self.update_descriptors()

            write("dataset.xyz", self.dataset)